# Stage 2 Data Augmentation

This notebook evaluates whether simple data-augmentation strategies improve Stage 2 dropout-point prediction when weekly sequences are flattened and modeled with LightGBM. It keeps only the augmentation experiments and the final comparison table from the older data-augmentation notebook.

The purpose is not to claim that augmentation always improves performance, but to test whether the model benefits from additional synthetic or perturbed training examples under a fixed evaluation protocol.


## Setup


In [3]:
import os
from collections import Counter
from pathlib import Path

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

import lightgbm as lgb

try:
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ModuleNotFoundError:
    SMOTE = None
    HAS_IMBLEARN = False

PROJECT_ROOT = Path.cwd()

from src.preprocess import build_stage2_dataset
from src import augmentation_utils as aug
from src import metrics_utils as metrics
from src import sequence_utils as seq

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
SEQUENCE_LENGTH = 12
MIN_WINDOW_LENGTH = 2
MODEL_NAME = "LightGBM"

print("model:", MODEL_NAME)
print("SMOTE available:", HAS_IMBLEARN)


model: LightGBM
SMOTE available: True


## Build Stage 2 Data

The weekly trajectory table is rebuilt through `build_stage2_dataset`, ensuring that the augmentation experiment uses the same feature definitions and dropout-point label as the Stage 2 sequence model.

Only the training data is augmented. Validation and test sets remain untouched, so the evaluation reflects performance on real observed learner trajectories.


In [4]:
stage2 = build_stage2_dataset(DATA_DIR)
df_full = stage2.df_full
feature_cols = stage2.feature_cols

print(f"timeline rows: {len(df_full):,}")
print(f"users: {df_full['user_id'].nunique():,}")
print(f"features: {len(feature_cols)}")
print(f"labeled rows: {df_full['is_dropout_point'].notna().sum():,}")


timeline rows: 408,377
users: 22,470
features: 17
labeled rows: 342,844


## Build Flattened Sequence Data

The old augmentation setup trains tabular classifiers on flattened weekly sequences. Each sequence is reshaped from `(weeks, features)` into one long feature vector.

For training, multi-window sequence construction creates additional examples from shorter histories padded to the fixed length. Validation and test sequences use the standard fixed history length, which keeps the comparison fair.


In [5]:
train_df, val_df, test_df, scaler = seq.split_and_scale_by_users(
    df_full,
    feature_cols,
    test_size=0.20,
    val_size=0.20,
    random_state=RANDOM_STATE,
)

X_train, y_train = seq.make_sequences_multi_window(
    train_df,
    feature_cols,
    sequence_length=SEQUENCE_LENGTH,
    min_window_length=MIN_WINDOW_LENGTH,
)
X_val, y_val = seq.make_sequences(
    val_df,
    feature_cols,
    sequence_length=SEQUENCE_LENGTH,
)
X_test, y_test = seq.make_sequences(
    test_df,
    feature_cols,
    sequence_length=SEQUENCE_LENGTH,
)

X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat = X_val.reshape(len(X_val), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

print("train:", X_train_flat.shape, Counter(y_train))
print("val:  ", X_val_flat.shape, Counter(y_val))
print("test: ", X_test_flat.shape, Counter(y_test))


train: (1848330, 204) Counter({np.float64(1.0): 925446, np.float64(0.0): 922884})
val:   (33754, 204) Counter({np.float64(0.0): 17204, np.float64(1.0): 16550})
test:  (43511, 204) Counter({np.float64(0.0): 22383, np.float64(1.0): 21128})


## Augmentation Methods

The notebook compares a non-augmented baseline with several augmentation strategies.

- SMOTE synthesizes minority-class examples from nearest neighbors, but is expensive on large flattened sequence matrices, so it is run on a stratified subset.
- Mixup interpolates between pairs of examples.
- Continuous noise perturbs numerical columns with small random variation.
- Masking replaces random values with column medians, simulating missing or weakened signals.

All methods are evaluated with the same LightGBM configuration.


In [6]:
# SMOTE is done on a stratified subset only. Running nearest-neighbor SMOTE
# on the full multi-window training matrix is prohibitively slow here.
augmentation_specs = {
    "baseline": lambda X, y: (X, y),
    "smote": lambda X, y: aug.smote_on_subset(
        X,
        y,
        max_smote_samples=200_000,
        random_state=RANDOM_STATE,
        smote_cls=SMOTE if HAS_IMBLEARN else None,
    ),
    "mixup": aug.mixup_augmentation,
    "cont_noise": aug.continuous_noise_augmentation,
    "masking": aug.masking_augmentation,
}


## Train And Evaluate All Augmentations

Each augmentation is applied to the training matrix, then a new LightGBM model is fit and evaluated. The decision threshold is selected on the validation set using F1, and the final metrics are reported on the untouched test set.

This design makes the comparison about the augmentation strategy rather than about different models or different thresholds.


In [7]:
def make_classifier():
    return lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbose=-1,
    )


def fit_classifier(model, X_train, y_train):
    return model.fit(
        X_train,
        y_train,
        eval_set=[(X_val_flat, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )


def evaluate_trained_model(name, model):
    y_val_proba = model.predict_proba(X_val_flat)[:, 1]
    best_threshold, best_val_f1 = metrics.best_threshold_by_f1(y_val, y_val_proba)

    y_test_proba = model.predict_proba(X_test_flat)[:, 1]
    y_test_pred = (y_test_proba >= best_threshold).astype(int)
    cm = confusion_matrix(y_test, y_test_pred)

    return {
        "augmentation": name,
        "classifier": MODEL_NAME,
        "best_threshold": best_threshold,
        "val_f1_at_threshold": best_val_f1,
        "test_auc": roc_auc_score(y_test, y_test_proba),
        "test_accuracy": accuracy_score(y_test, y_test_pred),
        "test_precision": precision_score(y_test, y_test_pred),
        "test_recall": recall_score(y_test, y_test_pred),
        "test_f1": f1_score(y_test, y_test_pred),
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
    }


models = {}
results = []

for name, augment in augmentation_specs.items():
    print("=" * 80)
    print(name)
    X_aug, y_aug = augment(X_train_flat, y_train)
    print("augmented train:", X_aug.shape, Counter(y_aug))

    model = make_classifier()
    fit_classifier(model, X_aug, y_aug)
    models[name] = model
    results.append(evaluate_trained_model(name, model))

results_df = pd.DataFrame(results).sort_values("test_f1", ascending=False)
results_df


baseline
augmented train: (1848330, 204) Counter({np.float64(1.0): 925446, np.float64(0.0): 922884})


c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


smote
SMOTE subset: (200000, 204), Counter({np.float64(1.0): 100138, np.float64(0.0): 99862})
augmented train: (200276, 204) Counter({np.float64(0.0): 100138, np.float64(1.0): 100138})


c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


mixup
augmented train: (2772495, 204) Counter({np.float64(1.0): 1388516, np.float64(0.0): 1383979})


...

cont_noise
continuous-noise columns: 156 / 204
augmented train: (2772495, 204) Counter({np.float64(1.0): 1389199, np.float64(0.0): 1383296})


c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


masking
augmented train: (2772495, 204) Counter({np.float64(1.0): 1389199, np.float64(0.0): 1383296})


c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,augmentation,classifier,best_threshold,val_f1_at_threshold,test_auc,test_accuracy,test_precision,test_recall,test_f1,tn,fp,fn,tp
3,cont_noise,LightGBM,0.422727,0.830530,0.891064,0.815725,0.748352,0.934873,0.831278,15741,6642,1376,19752
4,masking,LightGBM,0.431818,0.829958,0.890278,0.815610,0.749514,0.931607,0.830699,15805,6578,1445,19683
0,baseline,LightGBM,0.431818,0.829549,0.890609,0.815311,0.749239,0.931371,0.830436,15797,6586,1450,19678
2,mixup,LightGBM,0.431818,0.829700,0.889836,0.814805,0.748801,0.930897,0.829978,15785,6598,1460,19668
1,smote,LightGBM,0.431818,0.828158,0.888199,0.812507,0.747595,0.926780,0.827599,15772,6611,1547,19581


## Save Results

The saved table contains one row per augmentation strategy with the selected threshold, validation F1, test metrics, and confusion-matrix counts. This is the table to use when deciding whether an augmentation method is worth keeping.


In [8]:
results_df.to_csv(OUTPUT_DIR / "stage2_data_augmentation_results.csv", index=False)
results_df


,augmentation,classifier,best_threshold,val_f1_at_threshold,test_auc,test_accuracy,test_precision,test_recall,test_f1,tn,fp,fn,tp
3,cont_noise,LightGBM,0.422727,0.830530,0.891064,0.815725,0.748352,0.934873,0.831278,15741,6642,1376,19752
4,masking,LightGBM,0.431818,0.829958,0.890278,0.815610,0.749514,0.931607,0.830699,15805,6578,1445,19683
0,baseline,LightGBM,0.431818,0.829549,0.890609,0.815311,0.749239,0.931371,0.830436,15797,6586,1450,19678
2,mixup,LightGBM,0.431818,0.829700,0.889836,0.814805,0.748801,0.930897,0.829978,15785,6598,1460,19668
1,smote,LightGBM,0.431818,0.828158,0.888199,0.812507,0.747595,0.926780,0.827599,15772,6611,1547,19581
